# SEM-NAS — On-the-fly proxy experiments

Reference notebook for the paper *Budgeted Fixed-Proxy Search for Zero-Shot NAS on NAS-Bench-201 via Sample-Efficient Memetic NAS*.

This notebook walks through the **online** proxy mode:
every candidate produced by the search loop builds a real NAS-Bench-201 architecture and computes the chosen zero-cost proxy on a fixed minibatch via PyTorch.

We do **not** use the precomputed `nb201_<dataset>.pkl` lookup tables in this notebook.

**Sections**
1. Setup and imports
2. Build a single NB-201 architecture from a length-6 encoding
3. Compute one zero-cost proxy on it (online)
4. Spot-check all seven proxies on a hand-picked architecture
5. Run SEM-NAS at FFC = 100 with the online backend
6. Compare SEM-NAS to one budgeted fixed-proxy baseline (e.g., generic GA)
7. Mini-matrix sketch (small subset of the paper's 84 cells)

## 1. Setup and imports

If you cloned the repository at `code/`, run this notebook from `code/notebooks/` so the `sem_nas` package on `../` is importable. The cell below also adjusts `sys.path` defensively.

In [ ]:
import sys, time
from pathlib import Path

# Make the parent directory ('code/') importable regardless of how the
# notebook is launched.
REPO = Path.cwd().resolve()
while REPO.name != 'code' and REPO.parent != REPO:
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import numpy as np
import torch

from sem_nas.encoding import (
    N_ARCHS, N_EDGES, OPS, encoding_to_index, index_to_encoding,
)
from sem_nas.evaluator import FitnessEvaluator
from sem_nas.proxy import OnlineProxyBackend
from sem_nas.proxy.proxies import PROXY_NAMES
from sem_nas.proxy.nb201 import build_network
from sem_nas.sem_nas import run as run_sem_nas
from sem_nas.baselines import BASELINES

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, '| device:', DEVICE)
print('search space:', N_ARCHS, 'architectures, length-', N_EDGES, 'encoding')
print('proxies available:', PROXY_NAMES)

## 2. Build a single NB-201 architecture

An architecture is a length-6 integer vector with entries in `[0, 5)`. Each entry indexes into the operation set
`('none', 'skip_connect', 'nor_conv_1x1', 'nor_conv_3x3', 'avg_pool_3x3')`.

In [ ]:
# Pick an architecture: alternating 3x3 conv and skip-connect on the six edges.
encoding = np.array([3, 1, 3, 1, 3, 1], dtype=int)
print('encoding   :', encoding)
print('arch index :', encoding_to_index(encoding))
print('ops        :', [OPS[op] for op in encoding])

net = build_network(encoding, num_classes=10, cells_per_stage=2,
                    init_seed=0, device=DEVICE)
n_params = sum(p.numel() for p in net.parameters())
print(f'network parameters: {n_params:,}')

## 3. Compute one proxy online

The `OnlineProxyBackend` builds the architecture, initializes its weights, runs the chosen proxy on a cached minibatch, and returns one scalar.
Every call is independent: identical encoding + identical seeds give identical scores.

In [ ]:
backend = OnlineProxyBackend(
    proxy_name='zico',
    dataset='cifar10',
    batch_size=16,
    n_batches=2,            # ZiCo needs >= 2 minibatches
    cells_per_stage=2,
    device=DEVICE,
    data_source='random',   # no internet/torchvision needed
    init_seed=0,
    cell_seed=0,
)
t0 = time.time()
score = backend.evaluate(encoding)
print(f'ZiCo score = {score:.4f}  ({time.time() - t0:.2f}s)')

## 4. Spot-check all seven proxies on the same architecture

Each proxy uses the same minibatches but a freshly initialized network. The wall-clock cost shown is the time to build the architecture and run the proxy once — i.e., the cost of a single FFC unit in online mode.

In [ ]:
rows = []
for name in PROXY_NAMES:
    bk = OnlineProxyBackend(
        proxy_name=name,
        dataset='cifar10',
        batch_size=16,
        n_batches=2 if name == 'zico' else 1,
        cells_per_stage=2,
        device=DEVICE,
        data_source='random',
        init_seed=0,
        cell_seed=0,
        cache_results=False,
    )
    t0 = time.time()
    score = bk.evaluate(encoding)
    rows.append((name, score, time.time() - t0))

print(f'{"proxy":<10} {"score":>14} {"time (s)":>10}')
for name, score, dt in rows:
    print(f'{name:<10} {score:>14.4e} {dt:>10.3f}')

## 5. Run SEM-NAS at FFC = 100 with the online backend

Each evaluate call now drives a real NB-201 build plus a forward/backward pass. The wall-clock cost is dominated by these proxy computations rather than by the search loop.

We run SEM-NAS with the paper-final hyperparameters (`pop_size=10, K_gen=4, K_LS=3, W=3, b_LS=25`).

In [ ]:
# Trim cells_per_stage / batch_size to keep the notebook responsive on CPU.
# On GPU you can set cells_per_stage=5 and batch_size=64 for closer-to-paper
# proxy values without much wall-clock impact.

PROXY = 'synflow'    # data-free; fastest of the seven
DATASET = 'cifar10'
FFC = 100
SEED = 0

backend = OnlineProxyBackend(
    proxy_name=PROXY,
    dataset=DATASET,
    batch_size=16,
    n_batches=2,
    cells_per_stage=2,
    device=DEVICE,
    data_source='random',
    init_seed=SEED,
    cell_seed=SEED,
)
evaluator = FitnessEvaluator(backend, max_evals=FFC)
np.random.seed(SEED)

t0 = time.time()
best_arch, final_pop, final_fit = run_sem_nas(evaluator)
elapsed = time.time() - t0

print(f'SEM-NAS on {DATASET} / {PROXY}, FFC = {evaluator.ffc}')
print(f'  best encoding   : {best_arch.tolist()}')
print(f'  best proxy score: {max(evaluator.best_score_per_ffc):.4f}')
print(f'  wall time       : {elapsed:.1f}s '
      f'(~{elapsed / max(evaluator.ffc, 1) * 1000:.0f} ms / FFC)')

### Running best curve
The evaluator records the running best after each FFC, so the same run answers any FFC budget below `max_evals`.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6.5, 3.2))
ax.plot(np.arange(1, len(evaluator.best_score_per_ffc) + 1),
        evaluator.best_score_per_ffc, color='#d62728', linewidth=2.0,
        marker='*', markersize=4)
ax.set_xlabel('FFC')
ax.set_ylabel(f'best {PROXY} score (running)')
ax.set_title(f'SEM-NAS / {DATASET} / {PROXY} (online proxy compute, seed={SEED})')
ax.grid(True, linewidth=0.3, alpha=0.5)
plt.tight_layout()
plt.show()

## 6. SEM-NAS vs one baseline at the same FFC budget

Same proxy, same dataset, same seed; only the search loop differs.

In [ ]:
def run_method(method_name, proxy, dataset, ffc, seed):
    bk = OnlineProxyBackend(
        proxy_name=proxy, dataset=dataset, batch_size=16,
        n_batches=2 if proxy == 'zico' else 1,
        cells_per_stage=2, device=DEVICE,
        data_source='random', init_seed=seed, cell_seed=seed,
    )
    ev = FitnessEvaluator(bk, max_evals=ffc)
    np.random.seed(seed)
    if method_name == 'sem_nas':
        best, _, _ = run_sem_nas(ev)
    else:
        best = BASELINES[method_name](ev)
    return ev, best

results = {}
for method in ('sem_nas', 'generic_ga'):
    t0 = time.time()
    ev, best = run_method(method, PROXY, DATASET, FFC, SEED)
    results[method] = (ev, best, time.time() - t0)

print(f'{"method":<14} {"best proxy":>14} {"time (s)":>10}')
for m, (ev, best, dt) in results.items():
    print(f'{m:<14} {max(ev.best_score_per_ffc):>14.4f} {dt:>10.1f}')

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.2))
styles = {
    'sem_nas':   {'color': '#d62728', 'linestyle': '-',  'marker': '*', 'label': 'Proposed (SEM-NAS)'},
    'generic_ga': {'color': '#ff7f0e', 'linestyle': (0, (3, 1, 1, 1)),
                   'marker': 'D', 'label': 'Generic GA'},
}
for method, (ev, _, _) in results.items():
    ax.plot(np.arange(1, len(ev.best_score_per_ffc) + 1),
            ev.best_score_per_ffc, linewidth=1.8, markersize=4, **styles[method])
ax.set_xlabel('FFC')
ax.set_ylabel(f'best {PROXY} score (running)')
ax.set_title(f'SEM-NAS vs Generic GA / {DATASET} / {PROXY} (online compute, single seed)')
ax.grid(True, linewidth=0.3, alpha=0.5)
ax.legend(loc='lower right', frameon=True)
plt.tight_layout()
plt.show()

## 7. Mini-matrix sketch

A handful of cells in the same cell layout as the paper's 84-cell main matrix. We use a tiny seed count and a small subset of methods/proxies; bump `SEEDS`, `METHODS`, `PROXIES` to scale up. Every cell still triggers real PyTorch proxy computation per FFC.

In [ ]:
import itertools
import pandas as pd

METHODS = ('sem_nas', 'generic_ga')
PROXIES = ('synflow', 'snip')        # both are fast on CPU
DATASETS = ('cifar10',)
SEEDS = (0, 1, 2)
MINI_FFC = 60

rows = []
for method, dataset, proxy, seed in itertools.product(METHODS, DATASETS, PROXIES, SEEDS):
    ev, best = run_method(method, proxy, dataset, MINI_FFC, seed)
    rows.append({
        'method': method, 'dataset': dataset, 'proxy': proxy, 'seed': seed,
        'best_proxy': max(ev.best_score_per_ffc),
    })
df = pd.DataFrame(rows)
summary = df.groupby(['method', 'dataset', 'proxy'])['best_proxy'].agg(['mean', 'std']).reset_index()
summary

Increasing `SEEDS` to 100 and adding the rest of the methods/proxies scales this to the full 84-cell paper matrix. On a CPU laptop this is far slower than the precomputed-pickle path; consider a GPU and `multiprocessing` (see `scripts/run_main_matrix.py`).

---

**Tip — switching back to the precomputed lookup**

If you have the NAS-Bench-201 proxy pickles available locally, swap the backend in two lines:

```python
from sem_nas.proxy import PrecomputedProxyBackend
from scripts.load_proxy_pickle import load_proxy

scores, test_acc = load_proxy('cifar10', 'zico', data_dir='../data')
backend = PrecomputedProxyBackend(scores, test_acc)
evaluator = FitnessEvaluator(backend, max_evals=100)
```

The search loop is unchanged.